In [ ]:
import os
import sys
from pathlib import Path

# Get the notebook's directory
NOTEBOOK_DIR = Path(os.getcwd())
BACKEND_DIR = NOTEBOOK_DIR.parent.parent / "Skin_Lesion_Classification_backend"
ML_DIR = BACKEND_DIR / "ml"

# Add backend ml/ to sys.path so imports like `from src.data.dataset` work
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Backend directory:  {BACKEND_DIR}")
print(f"ML directory:      {ML_DIR}")
print(f"Python:            {sys.executable}")

# Verify backend exists
if not BACKEND_DIR.exists():
    print(f"\u274c ERROR: Backend not found at {BACKEND_DIR}")
    print("Make sure the backend repo is cloned at the same level as the frontend repo.")
else:
    print("\u2705 Backend directory found.")

# RQ3 — Does Backbone Architecture Affect XAI Quality?
## XAI Skin Lesion Classification — Research Question 3

**Hypothesis**: Models with similar classification accuracy (AUC) can produce
significantly different XAI quality because different architectures learn
different internal representations.

**What you're measuring**: XAI quality metrics (focus area, entropy) per backbone,
controlling for accuracy.

**Key concept**: You're separating *what the model knows* (accuracy) from
*how it explains itself* (XAI quality). These are different things — and that's the insight.

**Prerequisites**: You need all 3 trained models:
  - ResNet50
  - EfficientNet-B2
  - MobileNetV2

---

## CELL 1: Load All Three Trained Models

In [ ]:
import sys, os
from pathlib import Path

# ─── Self-contained setup ───
NOTEBOOK_DIR = Path(os.getcwd())
BACKEND_DIR = NOTEBOOK_DIR.parent.parent / "Skin_Lesion_Classification_backend"
ML_DIR = BACKEND_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

METADATA_PATH = ML_DIR / "data" / "processed" / "metadata_with_paths.csv"

MODEL_NAMES = ['resnet50', 'efficientnet_b2', 'mobilenetv2_100']
MODEL_PATHS = {name: ML_DIR / "outputs" / "models" / f"{name}_best.pth" for name in MODEL_NAMES}

missing = [n for n, p in MODEL_PATHS.items() if not p.exists()]
if missing:
    print(f"❌ Missing models: {missing}")
    print("Train models before running this notebook.")
    raise FileNotFoundError(f"Missing models: {missing}")

try:
    import torch
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import scipy.stats as stats
    from PIL import Image
    from tqdm import tqdm

    from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from src.models.classifier import create_model, get_target_layer
    from src.data.dataset import get_transforms, create_splits
    from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, classification_report

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    models = {}
    target_layers = {}

    for name in MODEL_NAMES:
        model = create_model(name, num_classes=2).to(device)
        model.load_state_dict(torch.load(MODEL_PATHS[name], map_location=device))
        model.eval()
        models[name] = model
        target_layers[name] = get_target_layer(model, name)
        print(f"Loaded {name}")

    transform = get_transforms('test', 224)

except ImportError as e:
    print(f"❌ Backend module not found: {e}")
    print("The backend ML code is not implemented yet.")
    print("Create: src/models/classifier.py and src/data/dataset.py")

---

## CELL 2: Compute Classification Metrics for Each Backbone

**WHY**: You need to establish that the models have SIMILAR accuracy
before comparing XAI quality. If one model is much better,
of course its XAI will look better — you'd be confounding
accuracy with XAI quality.

In [ ]:
# ─── Self-contained: check prerequisites ───
try:
    models
except NameError:
    print("❌ Models not loaded. Run CELL 2 first.")
    raise NameError("Run CELL 2 first.")

import pandas as pd

df = pd.read_csv(METADATA_PATH)
_, _, test_df = create_splits(df)

print("=== Classification Performance per Backbone ===\n")
backbone_performance = {}

for model_name, model in models.items():
    all_probs = []
    all_preds = []
    all_labels = []

    for _, row in test_df.iterrows():
        orig = np.array(Image.open(row['filepath']).convert('RGB').resize((224, 224)))
        input_t = transform(image=orig)['image'].unsqueeze(0).to(device)

        with torch.no_grad():
            out = model(input_t)
            prob = torch.softmax(out, dim=1)[0, 1].item()
            pred = int(prob >= 0.5)

        all_probs.append(prob)
        all_preds.append(pred)
        all_labels.append(int(row['label']))

    auc  = roc_auc_score(all_labels, all_probs)
    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds)

    backbone_performance[model_name] = {
        'auc': auc, 'acc': acc, 'f1': f1,
        'probs': all_probs, 'preds': all_preds, 'labels': all_labels
    }

    print(f"{model_name}:")
    print(f"  AUC: {auc:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")
    print(f"  {classification_report(all_labels, all_preds, target_names=['Benign','Malignant'], digits=3)}")

print("\nKEY QUESTION: Are the AUC differences small enough (< 0.05) that we can")
print("attribute XAI differences to architecture, not accuracy?")

aucs = [backbone_performance[m]['auc'] for m in MODEL_NAMES]
print(f"AUC range: {min(aucs):.4f} – {max(aucs):.4f} (diff = {max(aucs)-min(aucs):.4f})")

---

## CELL 3: Compute XAI Metrics for Each Backbone

**WHY**: Now that you know their accuracy is comparable, you can fairly compare XAI quality.

In [ ]:
# ─── Self-contained: check prerequisites ───
try:
    models
except NameError:
    print("❌ Models not loaded. Run CELL 2 first.")
    raise NameError("Run CELL 2 first.")

def focus_area_percentage(cam, threshold=0.5):
    return float((cam >= threshold).sum() / cam.size)

def cam_entropy(cam):
    cam_flat = cam.flatten()
    cam_flat = cam_flat / (cam_flat.sum() + 1e-8)
    return -np.sum(cam_flat * np.log(cam_flat + 1e-8))

rq3_results = []

# Use same images for all backbones for fair comparison
eval_df = test_df.sample(100, random_state=42)

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    orig = np.array(Image.open(row['filepath']).convert('RGB').resize((224, 224)))
    input_t = transform(image=orig)['image'].unsqueeze(0).to(device)

    for model_name, model in models.items():
        with torch.no_grad():
            out = model(input_t)
            probs = torch.softmax(out, dim=1)[0]
            pred = probs.argmax().item()
            conf = probs[pred].item()

        correct = int(pred == int(row['label']))

        for cam_method_name, cam_class in [('GradCAM', GradCAM), ('GradCAM++', GradCAMPlusPlus)]:
            with cam_class(model=model, target_layers=[target_layers[model_name]]) as cam_gen:
                cam = cam_gen(input_tensor=input_t, targets=[ClassifierOutputTarget(pred)])[0]

            rq3_results.append({
                'image_id':   row['image_id'],
                'dx':         row['dx'],
                'label':      row['label'],
                'model':      model_name,
                'cam_method': cam_method_name,
                'pred':       pred,
                'confidence': conf,
                'correct':    correct,
                'fap_05':     focus_area_percentage(cam, 0.5),
                'entropy':    cam_entropy(cam),
            })

rq3_df = pd.DataFrame(rq3_results)
OUTPUTS_DIR = NOTEBOOK_DIR / "outputs" / "metrics"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
rq3_df.to_csv(OUTPUTS_DIR / "RQ3_backbone_comparison.csv", index=False)
print(f"Results: {len(rq3_df)} rows")

---

## CELL 4: The Key Analysis — Does Backbone Predict XAI Quality?

In [ ]:
# ─── Self-contained: check prerequisites ───
try:
    rq3_df
except NameError:
    print("❌ Results not computed. Run CELL 6 first.")
    raise NameError("Run CELL 6 first.")

# Filter to correctly classified only
correct_df = rq3_df[rq3_df['correct'] == 1]

print("=== RQ3 — XAI Quality by Backbone (correct predictions only) ===\n")
print(correct_df.groupby(['model', 'cam_method'])[['fap_05', 'entropy']].mean().round(4))

# Statistical test: for each CAM method, are backbone differences significant?
for cam_method in ['GradCAM', 'GradCAM++']:
    sub = correct_df[correct_df['cam_method'] == cam_method]
    print(f"\n--- {cam_method} ---")

    groups = [sub[sub['model'] == m]['fap_05'].values for m in MODEL_NAMES]
    h, p = stats.kruskal(*groups)
    print(f"Kruskal-Wallis (FAP): H={h:.3f}, p={p:.4f} {'✅ significant' if p < 0.05 else '❌ not significant'}")

# Heatmap: backbone × cam_method
pivot = correct_df.groupby(['model', 'cam_method'])['fap_05'].mean().unstack()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('Focus Area % (RQ3)\nBackbone × CAM Method', fontsize=12)

# Bar chart
pivot.plot(kind='bar', ax=axes[1], colormap='Set2', edgecolor='black')
axes[1].set_title('Focus Area %: Does backbone change XAI quality?', fontsize=12)
axes[1].set_xlabel('Model Architecture')
axes[1].set_ylabel('Focus Area % (lower = more focused)')
axes[1].legend(title='CAM Method')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
OUTPUTS_DIR = NOTEBOOK_DIR / "outputs" / "figures"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(OUTPUTS_DIR / "RQ3_backbone_xai.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nKEY INSIGHT:")
print("If two models have similar AUC but different FAP/entropy,")
print("that means architecture affects HOW the model explains itself,")
print("not just how accurate it is — that's your paper contribution!")